# Step 1: Import all necessary libraries

In [39]:
# Libraries
import pandas as pd
from traffic.data import aircraft
import numpy as np
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)


## Step 2: Import existing bottleneck features

In [40]:
helper_bottleneck_features = pd.read_parquet('data/processed/helper_bottleneck_features.parquet')

In [41]:
helper_bottleneck_features.head()

,flight_id,taxi_start,taxi_end,pushback,sum_p10_hist_seg_time,sum_median_hist_seg_time,sum_p90_hist_seg_time,sum_seg_length_m,n_intersections_passed,TO_runway_28,TO_runway_32,TO_runway_16,TO_runway_10,TO_runway_34
0,GSW6KP_4832,2024-12-01 04:44:54+00:00,2024-12-01 04:58:20+00:00,1,NaN,NaN,NaN,711.4,4,0,1,0,0,0
1,EDW200E_4558,2024-12-01 04:58:35+00:00,2024-12-01 05:07:58+00:00,1,NaN,NaN,NaN,1190.3,7,0,1,0,0,0
2,EDW120_4456,2024-12-01 05:18:50+00:00,2024-12-01 05:25:38+00:00,1,NaN,NaN,NaN,740.8,4,0,1,0,0,0
3,EDW15T_4539,2024-12-01 05:31:07+00:00,2024-12-01 05:35:22+00:00,1,NaN,NaN,NaN,1092.5,7,0,1,0,0,0
4,EDW212R_4430,2024-12-01 05:34:06+00:00,2024-12-01 05:46:14+00:00,1,NaN,NaN,NaN,1911.9,11,0,1,0,0,0


## Step 3: Import processed trajectories

In [42]:
trajs_processed = pd.read_parquet('data/processed/trajs_processed.parquet')

In [43]:
trajs_processed.head()

,timestamp,icao24,latitude,longitude,groundspeed,track,vertical_rate,callsign,onground,alert,spi,squawk,altitude,geoaltitude,lastcontact,serials,hour,flight_id,track_unwrapped,cumdist,compute_gs,compute_track,registration,typecode,cut_runway,x,y,groundspeed_xy,track_xy,latitude_kalman,longitude_kalman,groundspeed_kalman,track_kalman,pushback
0,2024-12-03 19:27:23+00:00,501e43,47.454506,8.572392,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254042.608,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,281.121521,9A-JIM,C525,28,1835.107443,-394.409784,10.289896,288.756018,47.454498,8.572272,5.635465,298.075857,0
1,2024-12-03 19:27:24+00:00,501e43,47.454525,8.572311,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254043.929,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1829.065901,-392.298275,9.573292,288.535804,47.454504,8.572235,5.830707,303.050765,0
2,2024-12-03 19:27:25+00:00,501e43,47.454537,8.572259,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254043.929,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1825.082500,-391.005600,8.543856,295.514329,47.454516,8.572201,6.047056,307.521679,0
3,2024-12-03 19:27:26+00:00,501e43,47.454548,8.572206,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254045.899,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1821.099101,-389.712922,6.975231,305.511766,47.454536,8.572171,6.373721,315.379196,0
4,2024-12-03 19:27:27+00:00,501e43,47.454574,8.572181,<NA>,NaN,<NA>,9AJIM,True,False,False,1000,1200.0,<NA>,1733254045.899,[-1408231728],2024-12-03 19:00:00+00:00,9AJIM_8350,NaN,NaN,6.656234,327.148395,9A-JIM,C525,28,1819.240662,-386.836846,6.694868,316.065216,47.454562,8.572144,6.676477,321.610645,0


# Step 4: Import processed meteorological data

In [44]:
meteo_processed = pd.read_parquet('data/processed/meteo_processed.parquet')

## Step 5: Feature Engineering

In the following steps, additional features based on the processed trajectories are being created and then merged with the existing bottleneck features.

### Step 5.1: Operational Features

First, we create a features dataframe and define the target variable (taxi out time) for each flight_id.

In [45]:
features_df = helper_bottleneck_features[
    ["flight_id", "taxi_start", "taxi_end"]
].copy()

features_df["taxi_time_s"] = (
    features_df["taxi_end"] - features_df["taxi_start"]
).dt.total_seconds()

Before defining a filtering threshold, the distribution of taxi-out times is inspected using summary statistics and lower quantiles to understand the realistic range of values.

In [46]:
def inspect_taxi_time_distribution(features_df):

    print("Summary statistics:")
    print(features_df["taxi_time_s"].describe())

    print("\nLower quantiles:")
    print(
        features_df["taxi_time_s"].quantile(
            [0.01, 0.02, 0.05, 0.10]
        )
    )


inspect_taxi_time_distribution(features_df)

Summary statistics:
count    6.227200e+04
mean     1.675239e+04
std      4.092909e+05
min      3.000000e+00
25%      3.770000e+02
50%      5.620000e+02
75%      7.430000e+02
max      2.091277e+07
Name: taxi_time_s, dtype: float64

Lower quantiles:
0.01     41.0
0.02     78.0
0.05    134.0
0.10    196.0
Name: taxi_time_s, dtype: float64


A configurable minimum taxi-out time parameter is defined to allow flexible adjustment of the filtering threshold as the dataset grows or data characteristics change.


 <font color='red'> CHANGE THE PARAMETER IN THE CELL BELOW: </font>

In [47]:
MIN_TAXI_TIME_S = 60

Flights with taxi-out times below the defined minimum threshold are excluded from further analysis to remove incomplete or operationally implausible trajectories.

In [48]:
features_df = features_df[
    features_df["taxi_time_s"] >= MIN_TAXI_TIME_S
]

In [50]:
overnight_flights = features_df[
    features_df["taxi_start"].dt.date != features_df["taxi_end"].dt.date
]

overnight_flights[[
    "flight_id",
    "taxi_start",
    "taxi_end",
    "taxi_time_s"
]].sort_values("taxi_start")

,flight_id,taxi_start,taxi_end,taxi_time_s
2035,BAW719X_1498,2024-12-15 19:10:42+00:00,2025-02-20 18:19:02+00:00,5785700.0
2051,BAW707_1518,2024-12-15 20:14:29+00:00,2025-03-13 20:13:13+00:00,7603124.0
3957,SWR46M_8272,2024-12-29 20:11:53+00:00,2025-01-31 20:19:49+00:00,2851676.0
7012,SWR458E_3847,2025-01-21 11:17:31+00:00,2025-03-10 11:52:10+00:00,4149279.0
11237,BYD091_081,2025-02-21 20:17:54+00:00,2025-03-14 20:26:14+00:00,1814900.0
16977,IBE06DA_252,2025-04-02 17:39:54+00:00,2025-06-12 18:36:04+00:00,6137770.0
17289,DLH7C_1167,2025-04-04 18:06:48+00:00,2025-09-08 18:19:29+00:00,13565561.0
17678,IBE06SV_406,2025-04-07 05:20:29+00:00,2025-04-25 05:45:50+00:00,1556721.0
17796,CFG4310_975,2025-04-07 16:23:20+00:00,2025-05-04 16:10:47+00:00,2332047.0
17928,SWR5GM_4129,2025-04-08 15:23:08+00:00,2025-04-27 16:11:11+00:00,1644483.0


Adding aircraft typecode as a feature and setting datatype as "category"

In [ ]:
typecode_lookup = (
    trajs_processed[["flight_id", "typecode"]]
    .dropna(subset=["typecode"])
    .drop_duplicates()
)

features_df = features_df.merge(
    typecode_lookup,
    on="flight_id",
    how="left"
)

features_df["typecode"] = features_df["typecode"].astype("category")

The queue size feature is calculated within a rolling one-hour lookback window. For each flight, only aircraft that started taxiing within the previous hour and were still taxiing at the current flight's taxi_start time are counted. This keeps the feature operationally realistic, prediction-safe, and computationally scalable for larger datasets.

In [ ]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]

queue_size_last_hour = []

for i, t in enumerate(starts):
    n_active = (
        (starts >= t - WINDOW)
        & (starts < t)
        & (ends > t)
    ).sum()

    queue_size_last_hour.append(n_active)

features_df["queue_size"] = queue_size_last_hour

Implementing the average taxi out time of the past 6 hours as a feature

In [ ]:
WINDOW = pd.Timedelta(hours=6)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_6h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_6h.append(avg)

features_df["avg_taxi_out_time_6h"] = avg_taxi_out_6h

Implementing the average taxi out time of the past hour as a feature.

In [ ]:
WINDOW = pd.Timedelta(hours=1)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_1h = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_1h.append(avg)

features_df["avg_taxi_out_time_1h"] = avg_taxi_out_1h

Implementing the average taxi out time of the past 15 minutes as feature.

In [ ]:
WINDOW = pd.Timedelta(hours=0.25)

features_df = features_df.sort_values("taxi_start")

starts = features_df["taxi_start"]
ends = features_df["taxi_end"]
taxi_times = features_df["taxi_time_s"]

avg_taxi_out_15min = []

for i, t in enumerate(starts):

    mask = (
        (ends >= t - WINDOW)
        & (ends < t)
    )

    if mask.any():
        avg = taxi_times[mask].mean()
    else:
        avg = None

    avg_taxi_out_15min.append(avg)

features_df["avg_taxi_out_time_15min"] = avg_taxi_out_15min

Implementing bottleneck features

In [ ]:
cols_to_merge = [
    "flight_id",
    "n_intersections_passed",
    "sum_seg_length_m",
    "sum_p10_hist_seg_time",
    "sum_median_hist_seg_time",
    "sum_p90_hist_seg_time",
    "TO_runway_28",
    "TO_runway_32",
    "TO_runway_16",
    "TO_runway_10",
    "TO_runway_34",
    "pushback",
]

features_df = features_df.merge(
    helper_bottleneck_features[cols_to_merge],
    on="flight_id",
    how="left",
)

### Step 5.2 Temporal Features

Defining a function for cyclic encoding

In [ ]:
def sin_cos(x,T):
    return np.sin(2 * np.pi * x / T), np.cos(2 * np.pi * x / T)

Splitting up information from the 'taxi_start' column into month, day of week and minute of day

In [ ]:
features_df["month"] = features_df["taxi_start"].dt.month

features_df["day_of_week"] = features_df["taxi_start"].dt.dayofweek
# Monday = 0, Sunday = 6

features_df["minute_of_day"] = (
    features_df["taxi_start"].dt.hour * 60
    + features_df["taxi_start"].dt.minute
)

Encoding month, day of week and minute of day as sin and cos

In [ ]:
features_df["month_sin"], features_df["month_cos"] = sin_cos(
    features_df["month"], 12
)

features_df["day_of_week_sin"], features_df["day_of_week_cos"] = sin_cos(
    features_df["day_of_week"], 7
)

features_df["minute_of_day_sin"], features_df["minute_of_day_cos"] = sin_cos(
    features_df["minute_of_day"], 24 * 60
)

Adding a timestamp to the dataframe. This won't be used as a feautre but it will be used to define folds based on date

### Step 5.3: Meteorological Features

Implementing temperature and spread as  features

In [ ]:
meteo_temp = (
    meteo_processed[
        ["reference_timestamp", "tre200s0", "spread"]
    ]
    .rename(
        columns={
            "tre200s0": "temp_at_taxi_start",
            "spread": "spread_at_taxi_start"
        }
    )
)

features_df = pd.merge_asof(
    features_df.sort_values("taxi_start"),
    meteo_temp.sort_values("reference_timestamp"),
    left_on="taxi_start",
    right_on="reference_timestamp",
    direction="backward"
)

features_df = features_df.drop(columns="reference_timestamp")

In [ ]:
features_df.head(6)

Export features as parquet file

In [ ]:
output_path = "data/processed/features.parquet"
 
# Save dataframe to pkl.
features_df.to_parquet(output_path)
 
print(f"File \"features\" saved to: {output_path}")
 